In [38]:
import pandas as pd
import json
import openpyxl
import re
import sqlite3

In [39]:
df_arrivals_raw = pd.read_csv("track3_mandi_arrivals.csv")
df_master_raw = pd.read_csv("track3_mandi_master.csv")
df_transport_raw = pd.read_csv("track3_transport_logistics.csv")
df_weather_raw = pd.read_excel("track3_weather_sensors.xlsx")

with open("track3_price_and_msp.json", "r", encoding="utf-8") as f:
    price_json_raw = json.load(f)
df_price_raw = pd.DataFrame(price_json_raw)

raw_counts = {
    "mandi_arrivals": len(df_arrivals_raw),
    "mandi_master": len(df_master_raw),
    "transport_logistics": len(df_transport_raw),
    "weather_sensors": len(df_weather_raw),
    "price_and_msp": len(df_price_raw)
}
print("=== RAW DATASET ROW COUNTS ===")
for k, v in raw_counts.items():
    print(f"   • {k}: {v:,} raw rows")

print("\n=== DATASET COLUMNS SUMMARY ===")
print("Arrivals Columns:", df_arrivals_raw.columns.tolist())
print("Master Columns:", df_master_raw.columns.tolist())
print("Transport Columns:", df_transport_raw.columns.tolist())
print("Weather Columns:", df_weather_raw.columns.tolist())
print("Price Columns:", df_price_raw.columns.tolist())

=== RAW DATASET ROW COUNTS ===
   • mandi_arrivals: 25,750 raw rows
   • mandi_master: 60 raw rows
   • transport_logistics: 10,400 raw rows
   • weather_sensors: 15,000 raw rows
   • price_and_msp: 12,000 raw rows

=== DATASET COLUMNS SUMMARY ===
Arrivals Columns: ['arrival_id', 'date', 'mandi_id', 'crop_name', 'variety', 'arrival_quantity', 'unit', 'farmer_count']
Master Columns: ['mandi_id', 'mandi_name', 'district', 'state', 'mandi_type', 'total_area_acres']
Transport Columns: ['trip_id', 'mandi_id', 'destination_warehouse', 'departure_time', 'arrival_time', 'transit_hours', 'distance', 'distance_unit', 'vehicle_no', 'driver_id']
Weather Columns: ['sensor_id', 'timestamp', 'temperature', 'temp_unit', 'rainfall', 'rain_unit', 'humidity_percent']
Price Columns: ['record_id', 'date', 'mandi_id', 'district', 'crop_name', 'min_price', 'max_price', 'modal_price', 'msp']


In [40]:
df_arrivals = df_arrivals_raw.copy()
df_master = df_master_raw.copy()
df_price = df_price_raw.copy()
df_transport = df_transport_raw.copy()
df_weather = df_weather_raw.copy()

# Standardize Mandi IDs (MANDI001, MANDI-001, M001 -> MANDI_001)
def standardize_mandi_id(val):
    if pd.isna(val) or val is None:
        return None
    digits = re.sub(r"\D", "", str(val))
    if digits:
        return f"MANDI_{int(digits):03d}"
    return str(val).strip().upper()

for df in [df_arrivals, df_master, df_price]:
    m_col = [c for c in df.columns if 'mandi' in c.lower() and 'id' in c.lower()]
    if m_col:
        df[m_col[0]] = df[m_col[0]].apply(standardize_mandi_id)

CROP_MAP = {
    'wheat': 'Wheat', 'gehun': 'Wheat', 'gehu': 'Wheat', 'गेहूं': 'Wheat', 'kanak': 'Wheat',
    'rice': 'Rice', 'chawal': 'Rice', 'paddy': 'Rice', 'dhaan': 'Rice', 'basmati': 'Rice', 'धान': 'Rice', 'चावल': 'Rice',
    'mustard': 'Mustard', 'sarson': 'Mustard', 'sarso': 'Mustard', 'सरसों': 'Mustard',
    'sugarcane': 'Sugarcane', 'ganna': 'Sugarcane', 'ganne': 'Sugarcane', 'गन्ना': 'Sugarcane',
    'cotton': 'Cotton', 'kapas': 'Cotton', 'narma': 'Cotton', 'कपास': 'Cotton',
    'maize': 'Maize', 'corn': 'Maize', 'makki': 'Maize', 'makka': 'Maize', 'मक्का': 'Maize',
    'potato': 'Potato', 'aloo': 'Potato', 'आलू': 'Potato',
    'onion': 'Onion', 'pyaz': 'Onion', 'प्याज़': 'Onion',
    'tomato': 'Tomato', 'tamatar': 'Tomato', 'टमाटर': 'Tomato'
}

def clean_crop_name(val):
    if pd.isna(val) or val is None:
        return "Unknown"
    v = str(val).strip().lower()
    return CROP_MAP.get(v, str(val).strip().title())

for df in [df_arrivals, df_price]:
    c_col = [c for c in df.columns if 'crop' in c.lower() or 'comm' in c.lower()]
    if c_col:
        df['crop_name'] = df[c_col[0]].apply(clean_crop_name)

# Currency String Parser (Remove ₹, Rs., INR, commas)
def clean_currency_to_float(val):
    if pd.isna(val) or val is None:
        return None
    cleaned = re.sub(r"[₹RsINR,\s]", "", str(val), flags=re.IGNORECASE)
    try:
        return float(cleaned)
    except ValueError:
        return None

for col in ['modal_price', 'msp', 'min_price', 'max_price']:
    p_col = [c for c in df_price.columns if col in c.lower()]
    if p_col:
        df_price[p_col[0]] = df_price[p_col[0]].apply(clean_currency_to_float)
print("Mandi IDs, Crop names, and Currency strings standardized.")

Mandi IDs, Crop names, and Currency strings standardized.


In [43]:
# Parse quantities into Quintals (1 Tonne = 10 Qtl | 100 KG = 1 Qtl)
def parse_qty_to_quintals(val):
    if pd.isna(val) or val is None:
        return 0.0
    val_str = str(val).strip().lower()
    nums = re.findall(r"[\d.]+", val_str)
    if not nums:
        return 0.0
    amt = float(nums[0])
    if 'ton' in val_str:
        return amt * 10.0
    elif 'kg' in val_str:
        return amt / 100.0
    return amt

qty_cols = [c for c in df_arrivals.columns if ('quant' in c.lower() or 'qty' in c.lower() or 'vol' in c.lower() or 'qtl' in c.lower()) and 'date' not in c.lower()]
q_target = qty_cols[0] if qty_cols else df_arrivals.columns[2]
df_arrivals['arrival_quantity_qtl'] = df_arrivals[q_target].apply(parse_qty_to_quintals)

# Parse distances into KM (1 Mile = 1.60934 KM)
def parse_dist_to_km(val):
    if pd.isna(val) or val is None:
        return 0.0
    val_str = str(val).strip().lower()
    nums = re.findall(r"[\d.]+", val_str)
    if not nums:
        return 0.0
    dist = float(nums[0])
    if 'mile' in val_str or 'mi' in val_str.split():
        return round(dist * 1.60934, 2)
    return round(dist, 2)

d_cols = [c for c in df_transport.columns if 'dist' in c.lower() or 'km' in c.lower() or 'mile' in c.lower()]
d_target = d_cols[0] if d_cols else df_transport.columns[1]
df_transport['distance_km'] = df_transport[d_target].apply(parse_dist_to_km)

# Standardize Dates to IST YYYY-MM-DD
def parse_date_cleanly(series):
    parsed = pd.to_datetime(series, errors='coerce', utc=True, format='mixed')
    return parsed.dt.tz_convert('Asia/Kolkata').dt.date
for df in [df_arrivals, df_price, df_weather, df_transport]:
    dt_col = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower() or 'disp' in c.lower()]
    if dt_col:
        df['clean_date'] = parse_date_cleanly(df[dt_col[0]])
print("Quantities (Qtl), Distances (KM), and IST Dates (YYYY-MM-DD) parsed.")

Quantities (Qtl), Distances (KM), and IST Dates (YYYY-MM-DD) parsed.


In [44]:
# Temperature °F -> °C & Rain inches -> mm
def convert_temp_celsius(temp):
    if pd.isna(temp) or temp is None: return None
    try:
        val = float(temp)
        return round((val - 32) * 5.0 / 9.0, 2) if val > 45 else round(val, 2)
    except: return None

def convert_rain_mm(val):
    if pd.isna(val) or val is None: return None
    v_str = str(val).strip().lower()
    nums = re.findall(r"[\d.]+", v_str)
    if not nums: return None
    num_val = float(nums[0])
    return round(num_val * 25.4, 2) if 'inch' in v_str else round(num_val, 2)

t_col = [c for c in df_weather.columns if 'temp' in c.lower()]
r_col = [c for c in df_weather.columns if 'rain' in c.lower()]
if t_col: df_weather['temperature_celsius'] = df_weather[t_col[0]].apply(convert_temp_celsius)
if r_col: df_weather['rainfall_mm'] = df_weather[r_col[0]].apply(convert_rain_mm)

# Negative Transit Hours Fix & Vehicle Reg Formatting
def fix_transit_hours(val):
    if pd.isna(val) or val is None: return 0.0
    nums = re.findall(r"[\d.]+", str(val))
    return abs(float(nums[0])) if nums else 0.0

def format_veh_reg(reg):
    if pd.isna(reg) or reg is None: return "UNKNOWN"
    clean_reg = re.sub(r"[^A-Za-z0-9]", "", str(reg)).upper()
    return f"{clean_reg[:2]}-{clean_reg[2:4]}-{clean_reg[4:6]}-{clean_reg[6:]}" if len(clean_reg) == 10 else clean_reg

tr_col = [c for c in df_transport.columns if 'transit' in c.lower() or 'time' in c.lower() or 'hour' in c.lower()]
v_col = [c for c in df_transport.columns if 'veh' in c.lower() or 'reg' in c.lower()]
if tr_col: df_transport['clean_transit_hours'] = df_transport[tr_col[0]].apply(fix_transit_hours)
if v_col: df_transport['clean_vehicle_number'] = df_transport[v_col[0]].apply(format_veh_reg)

print("Weather sensors (°C / mm) & Transport anomalies fixed.")

Weather sensors (°C / mm) & Transport anomalies fixed.


In [45]:
# 1. DEDUPLICATE CLEANED DATAFRAMES
df_arrivals_clean = df_arrivals.drop_duplicates().reset_index(drop=True)
df_master_clean = df_master.drop_duplicates().reset_index(drop=True)
df_price_clean = df_price.drop_duplicates().reset_index(drop=True)
df_weather_clean = df_weather.drop_duplicates().reset_index(drop=True)
df_transport_clean = df_transport.drop_duplicates().reset_index(drop=True)

# 2. DATA CLEANING SUMMARY TABLE
data_cleaning_summary = pd.DataFrame([
    {
        'Dataset': 'Mandi Arrivals',
        'Raw Rows': raw_counts['mandi_arrivals'],
        'Clean Rows': len(df_arrivals_clean)
    },
    {
        'Dataset': 'Mandi Master',
        'Raw Rows': raw_counts['mandi_master'],
        'Clean Rows': len(df_master_clean)
    },
    {
        'Dataset': 'Price & MSP',
        'Raw Rows': raw_counts['price_and_msp'],
        'Clean Rows': len(df_price_clean)
    },
    {
        'Dataset': 'Weather Sensors',
        'Raw Rows': raw_counts['weather_sensors'],
        'Clean Rows': len(df_weather_clean)
    },
    {
        'Dataset': 'Transport Logistics',
        'Raw Rows': raw_counts['transport_logistics'],
        'Clean Rows': len(df_transport_clean)
    }
])

print("============ DATA CLEANING SUMMARY ============")
print(data_cleaning_summary.to_string(index=False))
print("\n")

# 3. EXPORT CLEANED DATASETS TO CSV
df_arrivals_clean.to_csv("clean_mandi_arrivals.csv", index=False)
df_master_clean.to_csv("clean_mandi_master.csv", index=False)
df_price_clean.to_csv("clean_price_and_msp.csv", index=False)
df_weather_clean.to_csv("clean_weather_sensors.csv", index=False)
df_transport_clean.to_csv("clean_transport_logistics.csv", index=False)

print("All 5 cleaned datasets exported to CSV files successfully!")

============ DATA CLEANING SUMMARY ============
            Dataset  Raw Rows  Clean Rows
     Mandi Arrivals     25750       25000
       Mandi Master        60          57
        Price & MSP     12000       12000
    Weather Sensors     15000       15000
Transport Logistics     10400       10000


All 5 cleaned datasets exported to CSV files successfully!


In [46]:
# Re-sync clean dataframes with updated columns
df_arrivals_clean = df_arrivals.drop_duplicates().reset_index(drop=True)
df_transport_clean = df_transport.drop_duplicates().reset_index(drop=True)

# Connect to SQLite Database
conn = sqlite3.connect("agritech_mandi.db")

# Save cleaned tables into SQLite
df_arrivals_clean.to_sql("mandi_arrivals", conn, if_exists="replace", index=False)
df_master_clean.to_sql("mandi_master", conn, if_exists="replace", index=False)
df_price_clean.to_sql("price_and_msp", conn, if_exists="replace", index=False)
df_weather_clean.to_sql("weather_sensors", conn, if_exists="replace", index=False)
df_transport_clean.to_sql("transport_logistics", conn, if_exists="replace", index=False)

print("Clean datasets loaded into SQLite Database ('agritech_mandi.db').")

# KPI 1: Total Crop Arrivals by Crop Type (Quintals)
kpi_arrivals_by_crop = pd.read_sql_query("""
    SELECT 
        crop_name, 
        ROUND(SUM(arrival_quantity_qtl), 2) AS total_arrival_qtl
    FROM mandi_arrivals
    WHERE crop_name IS NOT NULL AND crop_name != 'Unknown'
    GROUP BY crop_name
    ORDER BY total_arrival_qtl DESC
""", conn)

# KPI 2: Price Crash Instances (Modal Price < MSP Alert)
modal_col = [c for c in df_price_clean.columns if 'modal' in c.lower()][0]
msp_col = [c for c in df_price_clean.columns if 'msp' in c.lower()][0]
kpi_price_crashes = pd.read_sql_query(f"""
    SELECT 
        crop_name,
        clean_date,
        {modal_col} AS modal_price,
        {msp_col} AS msp,
        ROUND(({msp_col} - {modal_col}), 2) AS price_loss_below_msp
    FROM price_and_msp
    WHERE {modal_col} IS NOT NULL AND {msp_col} IS NOT NULL AND {modal_col} < {msp_col}
    ORDER BY price_loss_below_msp DESC
""", conn)

# KPI 3: Transit Delay Rate & Average Transit Hours by Warehouse
w_cols = [c for c in df_transport_clean.columns if 'ware' in c.lower() or 'dest' in c.lower() or 'wh' in c.lower()]
w_target = w_cols[0] if w_cols else df_transport_clean.columns[0]
kpi_warehouse_logistics = pd.read_sql_query(f"""
    SELECT 
        {w_target} AS destination_warehouse,
        COUNT(*) AS total_trips,
        ROUND(AVG(clean_transit_hours), 2) AS avg_transit_hours,
        SUM(CASE WHEN clean_transit_hours > 24 THEN 1 ELSE 0 END) AS delayed_trips,
        ROUND(SUM(CASE WHEN clean_transit_hours > 24 THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 2) AS delay_rate_pct
    FROM transport_logistics
    WHERE {w_target} IS NOT NULL
    GROUP BY {w_target}
    ORDER BY avg_transit_hours DESC
""", conn)

# Display Analytics Results
print("\n=== CORE KPI 1: TOTAL ARRIVALS BY CROP (QUINTALS) ===")
print(kpi_arrivals_by_crop.to_string(index=False))

print(f"\n=== CORE KPI 2: PRICE CRASH ALERT SUMMARY (MODAL PRICE < MSP) ===")
print(f"Total Price Crash Incidents Detected: {len(kpi_price_crashes):,} records")
print(kpi_price_crashes.head(5).to_string(index=False))

print("\n=== CORE KPI 3: WAREHOUSE TRANSIT DELAY METRICS ===")
print(kpi_warehouse_logistics.to_string(index=False))

Clean datasets loaded into SQLite Database ('agritech_mandi.db').

=== CORE KPI 1: TOTAL ARRIVALS BY CROP (QUINTALS) ===
crop_name  total_arrival_qtl
    Wheat        28743046.91
  Mustard        28485204.42
Sugarcane        28239705.26
   Cotton        27425319.41
     Rice        26094871.61
    Maize        25857993.60

=== CORE KPI 2: PRICE CRASH ALERT SUMMARY (MODAL PRICE < MSP) ===
Total Price Crash Incidents Detected: 3,185 records
crop_name clean_date  modal_price    msp  price_loss_below_msp
   Cotton 2026-04-06       0.5874 6620.0               6619.41
   Cotton 2026-08-22       0.5903 6620.0               6619.41
   Cotton 2026-03-06       0.5941 6620.0               6619.41
   Cotton 2026-06-03       0.5934 6620.0               6619.41
   Cotton 2026-04-06       0.5986 6620.0               6619.40

=== CORE KPI 3: WAREHOUSE TRANSIT DELAY METRICS ===
destination_warehouse  total_trips  avg_transit_hours  delayed_trips  delay_rate_pct
           WH-Central         1660       